In [ ]:
from datasets import load_dataset

csv_path = "/Users/Samuel/Desktop/SAMUEL MacBook/Samuel Yale/Courses/Thesis/simplegraphs/0-rels-no-gp.csv"

dataset = load_dataset(
        "csv", data_files=[csv_path], split="train", delimiter=",", column_names=["title", "text"]
    )

Generating train split: 81 examples [00:00, 1296.35 examples/s]


In [98]:
import torch
torch.set_grad_enabled(False)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print('CUDA')
elif torch.backends.mps.is_available():
    if torch.backends.mps.is_built():
        device =torch.device('mps')
        print('MPS')
else:
    device = torch.device("cpu")
    print('CPU')

MPS


In [99]:
from transformers import (
    DPRContextEncoder,
    DPRContextEncoderTokenizer,
)

dpr_ctx_encoder_model_name = "facebook/dpr-ctx_encoder-multiset-base"

dpr_tokenizer = DPRContextEncoderTokenizer.from_pretrained(dpr_ctx_encoder_model_name)
dpr_encoder = DPRContextEncoder.from_pretrained(dpr_ctx_encoder_model_name)

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'DPRQuestionEncoderTokenizer'. 
The class this function is called from is 'DPRContextEncoderTokenizer'.
Some weights of the model checkpoint at facebook/dpr-ctx_encoder-multiset-base were not used when initializing DPRContextEncoder: ['ctx_encoder.bert_model.pooler.dense.bias', 'ctx_encoder.bert_model.pooler.dense.weight']
- This IS expected if you are initializing DPRContextEncoder from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DPRContextEncoder from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification mode

In [100]:
import numpy as np

def testembed(docs: dict, encoder: DPRContextEncoder = dpr_encoder, tokenizer: DPRContextEncoderTokenizer = dpr_tokenizer):
    input_ids = tokenizer(docs['title'], docs['text'], return_tensors='pt')['input_ids']
    embeddings = encoder(input_ids, return_dict = True).pooler_output
    return {'embeddings': embeddings[0]}

In [101]:
from functools import partial

embedded = dataset.map(partial(testembed))

Map: 100%|██████████| 81/81 [00:03<00:00, 24.36 examples/s]


In [102]:
import os

context_dir = '/Users/Samuel/Desktop/SAMUEL MacBook/Samuel Yale/Courses/Thesis/Datasets/tests/contexts'
ver = 1
passages_path = os.path.join(context_dir, f"my_knowledge_dataset_{ver}")

embedded.save_to_disk(passages_path)

Saving the dataset (1/1 shards): 100%|██████████| 81/81 [00:00<00:00, 22818.10 examples/s]


In [103]:
# from datasets import load_from_disk
# dataset = load_from_disk(passages_path)

In [104]:
import faiss

index = faiss.IndexHNSWFlat(768, 128, faiss.METRIC_INNER_PRODUCT)
embedded.add_faiss_index("embeddings", custom_index=index)

100%|██████████| 1/1 [00:00<00:00, 105.33it/s]


Dataset({
    features: ['title', 'text', 'embeddings'],
    num_rows: 81
})

In [105]:
index_dir = '/Users/Samuel/Desktop/SAMUEL MacBook/Samuel Yale/Courses/Thesis/Datasets/tests/indices'
index_path = os.path.join(index_dir, f"my_knowledge_dataset_{ver}_hnsw_index.faiss")

embedded.get_index("embeddings").save(index_path)

In [106]:
# dataset.load_faiss_index("embeddings", index_path)

In [107]:
device = torch.device('mps')

In [108]:
from transformers import RagTokenizer, RagRetriever, RagTokenForGeneration

rag_name = 'facebook/rag-token-nq'

retriever = RagRetriever.from_pretrained(rag_name, index_name='custom', indexed_dataset=embedded) # n_docs
model = RagTokenForGeneration.from_pretrained(rag_name, retriever=retriever).to(device)
tokenizer = RagTokenizer.from_pretrained(rag_name)

/Users/Samuel/Desktop/SAMUEL MacBook/Samuel Yale/Courses/Thesis/senior-thesis-2/lib/python3.13/site-packages/transformers/models/bart/configuration_bart.py:176: UserWarning: Please make sure the config includes `forced_bos_token_id=0` in future versions. The config can simply be saved and uploaded again to be fixed.
  warnings.warn(
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'RagTokenizer'. 
The class this function is called from is 'DPRQuestionEncoderTokenizer'.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'RagTokenizer'. 
The class this function is called from is 'DPRQuestionEncoderTokenizerFast'.
The tokenizer class you load from this checkpoint is not the 

In [109]:
# question = "Who is the aunt of Aurora"
# input_ids = tokenizer.question_encoder(question, return_tensors='pt')['input_ids'].to(device)
# outputs = model.generate(input_ids=input_ids, device=device)
# out_string = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
# print(out_string)

In [110]:
def call(question:str) -> str:
    input_ids = tokenizer.question_encoder(question, return_tensors='pt')['input_ids'].to(device)
    outputs = model.generate(input_ids=input_ids, device=device)
    out_string = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    return out_string

def eval(question:str, answer:str):
    response = call(question)
    if answer.lower() in response.lower():
        return True
    else:
        print(question, answer, response)
        return False

In [ ]:
import csv
questions_path = '/Users/Samuel/Desktop/SAMUEL MacBook/Samuel Yale/Courses/Thesis/simplegraphs/0-questions-grandparents.csv'
results = []
with open(questions_path, mode='r') as file:
    reader = csv.reader(file)
    for row in reader:
        question = row[0]
        answer = row[1]
        results.append(eval(question, answer))

Who is the daughter of Addison? Luna  liam 's daughter
Who is the daughter of Emily? Addison  sebastian
Who is the daughter of Mia? Grace  aria
Who is the father of Mia? Connor  sebastian
Who is the grandfather of David? Connor  liam
Who is the grandfather of Elijah? Liam  connor
Who is the grandfather of Grace? Liam  connor
Who is the grandfather of Luna? Connor  liam
Who is the grandmother of David? Aria  emily
Who is the grandmother of Elijah? Aria  emily
Who is the grandmother of Grace? Emily  aria
Who is the grandmother of Luna? Emily  aria
Who is the mother of David? Addison  emily
Who is the mother of Grace? Mia  sebastian
Who is the sibling of Aria? Connor  mia
Who is the sibling of Connor? Aria  henry
Who is the sibling of David? Luna  sebastian
Who is the sibling of Emily? Liam  addison
Who is the sibling of Grace? Elijah  hismia
Who is the sibling of Liam? Emily  sebastian
Who is the son of Addison? David  liam
Who is the son of Aria? Henry  mia
Who is the son of Henry? Davi

In [115]:
print('Single-Doc Retrieval Accuracy:', sum(results)/len(results))

Single-Doc Retrieval Accuracy: 0.6875
